[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geo-di-lab/emerge-geoai-es/blob/main/docs/ch2/lesson3.ipynb)

# Mapear datos de ciencia participativa de GLOBE

Visualiza en mapas los datos de GLOBE Mosquito Habitat Mapper y GLOBE Land Cover.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap

mosquito = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/raw/"
    "refs/heads/main/docs/data/globe_mosquito.zip"
)

land_cover = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/raw/"
    "refs/heads/main/docs/data/globe_land_cover.zip"
)

Carga el límite de Florida, obtenido a partir del archivo de límites estatales del [Censo de Estados Unidos](https://www.census.gov/geographies/mapping-files/time-series/geo/cartographic-boundary.html) y filtrado previamente para incluir solamente Florida. Utilizaremos este límite para recortar los datos y concentrarnos en el estado.

In [ ]:
fl = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/raw/"
    "refs/heads/main/docs/data/florida_boundary.geojson"
)[["geometry"]]

mosquito = (
    gpd.sjoin(
        mosquito,
        fl,
        how="inner",
        predicate="intersects"
    )
    .drop(columns=["index_right"])
    .reset_index(drop=True)
)

land_cover = (
    gpd.sjoin(
        land_cover,
        fl,
        how="inner",
        predicate="intersects"
    )
    .drop(columns=["index_right"])
    .reset_index(drop=True)
)

Representa en un mapa los datos de mosquitos recopilados en Florida entre 2018 y 2024.

In [ ]:
# Mostrar las columnas disponibles
mosquito.columns

In [ ]:
# Crear un mapa vacío centrado en Florida
map = folium.Map(
    location=[28.263363, -83.497652],
    tiles="Cartodb dark_matter",
    zoom_start=7
)

# Agregar cada observación como un marcador
for idx, row in mosquito.iterrows():
    popup_content = (
        f"<b>Fecha:</b> {row['MeasuredDate']}<br>"
        f"<b>Fuente de agua:</b> {row['WaterSourceType']}<br>"
        f"<b>Longitud:</b> {row['MeasurementLongitude']}<br>"
        f"<b>Latitud:</b> {row['MeasurementLatitude']}"
    )

    folium.Marker(
        location=[
            row.geometry.y,
            row.geometry.x
        ],
        popup=folium.Popup(
            popup_content,
            max_width=300
        )
    ).add_to(map)

display(map)

En el mapa, haz clic en uno de los puntos para abrir una ventana con la fecha, el tipo de fuente de agua, la longitud y la latitud de la observación enviada mediante la aplicación GLOBE Observer.

Representa en un mapa los datos de cobertura terrestre recopilados entre 2018 y 2024.

In [ ]:
# Mostrar las columnas disponibles
land_cover.columns

In [ ]:
# Crear un mapa vacío centrado en Florida
map = folium.Map(
    location=[28.263363, -83.497652],
    tiles="Cartodb dark_matter",
    zoom_start=7
)

# Agregar cada observación como un marcador
for idx, row in land_cover.iterrows():
    popup_content = (
        f"<b>Fecha:</b> {row['MeasuredDate']}<br>"
        f"<b>MUC:</b> {row['MucDescription']}<br>"
        f"<b>Longitud:</b> {row['MeasurementLongitude']}<br>"
        f"<b>Latitud:</b> {row['MeasurementLatitude']}"
    )

    folium.Marker(
        location=[
            row.geometry.y,
            row.geometry.x
        ],
        popup=folium.Popup(
            popup_content,
            max_width=300
        )
    ).add_to(map)

display(map)

Hemos creado mapas interactivos sencillos con los datos de GLOBE. En la próxima lección crearemos mapas de calor que mostrarán la distribución mundial de estas observaciones.